#### RAG and Agent Evaluation

In [1]:
# LLM as a judge
import pandas as pd

df_ground_truth = pd.read_csv("../../../data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

/var/home/alexis/src/llm-zoomcamp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join the course. If you want a certificate, you need to submit your project while submissions are still being accepted.'

In [7]:
assistant.total_cost()

0.000507

In [8]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [9]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Can I still join the course if I just found it now?',
 'answer_llm': 'Yes, you can still join the course. If you want a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [10]:
# Processing all questions
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [11]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Can I still join the course if I just found it now?',
 'answer_llm': 'Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [12]:
assistant.reset_usage()

In [13]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [14]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

100%|██████████| 565/565 [02:45<00:00,  3.42it/s]


In [15]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [16]:
assistant.total_cost()

0.6217132499999998

In [17]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("../../../data/rag-answers-new.csv", index=False)

#### LLM as a Judge

In [18]:
import pandas as pd

df_answers = pd.read_csv("../../../data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [19]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [20]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [21]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [22]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [23]:
rec = answers[0]

In [24]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [25]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer exactly matches the ground truth and preserves the full meaning, including the certificate requirement tied to submission timing.', score='good')

In [26]:
calc_price(usage)

{'input_cost': 0.000216,
 'output_cost': 0.00017549999999999998,
 'total_cost': 0.0003915}

In [27]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [28]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth exactly in meaning and wording. It correctly states that joining is still possible, but certificate eligibility depends on submitting the project while submissions are still open.', score='good')

#### Running the judge

In [29]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [30]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

100%|██████████| 565/565 [09:06<00:00,  1.03it/s]


In [31]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [32]:
df_eval = pd.DataFrame(evaluations)

In [33]:
calc_total_price(usages)

0.399705

In [34]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 543/565 = 96.11%


In [35]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
3,What do I need to do to be eligible for the ce...,74eb249bbf,bad,The ground truth says that joining now is allo...
8,"I signed up for the course, but do I still nee...",977bf7786c,bad,The AI answer does not convey the ground truth...
34,What do I actually need to finish to get the c...,9f689c185f,bad,The ground truth says only that passing the Ca...
124,Could this be happening because I named the AP...,86d99bbf21,bad,The AI answer is not semantically equivalent t...
162,What’s the best way to keep my OpenAI key in a...,8b2f5e9d04,bad,The AI answer gives a reasonable way to load a...


In [36]:
df_eval.to_csv("../../../data/rag-evaluations-new.csv", index=False)

#### Agent Evaluation

In [37]:
import pandas as pd

df_ground_truth = pd.read_csv("../../../data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [38]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [39]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [40]:
# Running the agent
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [41]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [42]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [43]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [44]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Can I still join the course if I just found it now?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"join course late found now enrollment can I still join"}', call_id='call_JKuOtL7EXaPzkpSer9lrQCP3', name='search', type='function_call', id='fc_0060df50217e3097006a5639bf1c8481a294819ebc88485868', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_JKuOtL7EXaPzkpSer9lrQCP3',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still ac

In [45]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [46]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"join course late found now enrollment can I still join"}'}]

In [47]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [48]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'Can I still join the course if I just found it now?',
 'answer_agent': 'Yes — you can still join the course if you just found it now.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"join course late found now enrollment can I still join"}'}],
 'cost': Decimal('0.00093225'),
 'document': '74eb249bbf'}

In [49]:
# Processing multiple questions
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [50]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

100%|██████████| 50/50 [00:24<00:00,  2.02it/s]


In [51]:
df_agent = pd.DataFrame(agent_answers)


In [52]:
df_agent["cost"].sum()

Decimal('0.06294750')

In [54]:
df_agent.to_csv("../../../data/agent-answers.csv", index=False)

In [2]:
import pandas as pd
df_agent = pd.read_csv("../../../data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

#### Judging answers and trajectories

In [14]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [15]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [16]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [24]:
import ast
import json

from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        try:
            tool_calls = json.loads(tool_calls)
        except json.JSONDecodeError:
            tool_calls = ast.literal_eval(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [25]:
agent_answers[0]

{'question': 'Can I still join the course if I just found it now?',
 'answer_agent': 'Yes — you can still join the course.\n\nIf you want to receive a certificate, though, you’ll need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{\'name\': \'search\', \'arguments\': \'{"query":"join course late found it now late enrollment can I still join"}\'}]',
 'cost': 0.00091725,
 'document': '74eb249bbf'}

In [26]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent answer matches the ground truth: it says the user can still join the course, and that to receive a certificate they must submit the project while submissions are still open. This preserves the key information from the original answer.', answer_score='good', trajectory_reasoning='The single search query was relevant and included the main intent of the question about joining the course after finding it late. One tool call is reasonable here, and there were no unnecessary duplicate searches. The tool call supports the final answer.', trajectory_score='good')

In [27]:
# Running the agent judge
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [28]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

100%|██████████| 50/50 [00:23<00:00,  2.11it/s]


In [29]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [30]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [31]:
calc_total_price(usages)

0.05373675000000001

In [32]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    48
bad      2
Name: count, dtype: int64

In [33]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64

In [35]:
df_agent_eval.to_csv("../../../data/agent-evaluations.csv", index=False)